<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h1 style="margin-top:0; color:#1d4ed8; font-size:34px;">
🔧 02 Merge and Feature Engineering
</h1>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook transforms the raw <b>OULAD relational tables</b> into a structured, machine-learning-ready dataset.
</p>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The main objective is to combine <b>student demographics</b>, course information, VLE engagement, assessment performance, submission behavior, and workload indicators into one clean flat table.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🎯 Final Dataset Granularity
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<p style="font-size:16px; margin-top:0; color:#1e40af;"><b>One row per:</b></p>

<ul style="font-size:16px; line-height:2; margin-bottom:0;">
  <li><code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">id_student</code></li>
  <li><code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">code_module</code></li>
  <li><code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">code_presentation</code></li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This structure prevents <b>event-level duplication</b> and creates a reliable student-course-presentation dataset for early-warning prediction, student risk analysis, and later modeling experiments.
</p>

</div>

<h2 style="color:#EA580C;">📦 Import Libraries and Define Constants</h2>

In [1]:
import pandas as pd
import numpy as np

# Define a reference semester length used later
# for normalized time mapping experiments.
ARAB_SEMESTER_LENGTH = 150

<h2 style="color:#EA580C;">📥 Load Raw OULAD Tables</h2>

This section loads the main raw OULAD tables required for merging and feature engineering.

In [2]:
# Define the path containing the raw OULAD CSV files.
DATA_PATH = "C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\data\\raw\\"

# Load the main OULAD tables.
student_info = pd.read_csv(DATA_PATH + "studentInfo.csv")
courses = pd.read_csv(DATA_PATH + "courses.csv")
student_vle = pd.read_csv(DATA_PATH + "studentVle.csv")
vle = pd.read_csv(DATA_PATH + "vle.csv")
assessments = pd.read_csv(DATA_PATH + "assessments.csv")
student_assessment = pd.read_csv(DATA_PATH + "studentAssessment.csv")

<h2 style="color:#EA580C;">🧱 Create the Base Student Table</h2>

This section prepares the main student-level base table by filtering extreme credit values, creating the prediction target, and adding course presentation length.

In [3]:
# Start from student_info because it contains one row per student-course-presentation attempt.
base = student_info.copy()

# Remove extreme studied_credits values to reduce the impact of outliers.
base = base[base["studied_credits"] <= 250].copy()

# Create a binary prediction target:
# successful = Pass or Distinction
# at_risk = Fail or Withdrawn
base["target"] = base["final_result"].map({
    "Distinction": "successful",
    "Pass": "successful",
    "Fail": "at_risk",
    "Withdrawn": "at_risk"
})

# Add course-level metadata, especially module_presentation_length.
base = base.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

# Display the shape of the prepared base table.
base.shape

(32480, 14)

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The base table now contains student demographic information, academic background, final outcome, target label, and course presentation length.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
After filtering extreme 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">studied_credits</code> 
values, the table contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course-presentation records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">14</code> 
columns.
</p>

</div>

<h2 style="color:#EA580C;">🖱️ Prepare VLE Interaction Events</h2>

This section enriches student VLE interactions with activity metadata and course duration, then normalizes interaction dates by course progress.

In [4]:
# Merge student VLE interactions with VLE metadata.
# This adds activity_type and planned week information for each learning resource.
student_vle_full = student_vle.merge(
    vle,
    on=["id_site", "code_module", "code_presentation"],
    how="left"
)

# Add course presentation length to normalize interaction dates.
student_vle_full = student_vle_full.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

# Convert raw interaction day into relative course progress.
# Example: 0.25 means the interaction happened around the first 25% of the course.
student_vle_full["relative_time"] = (
    student_vle_full["date"] / student_vle_full["module_presentation_length"]
)

# Map relative course progress to an Arab-semester-style 150-day timeline.
student_vle_full["arab_semester_day"] = (
    student_vle_full["relative_time"] * ARAB_SEMESTER_LENGTH
)

# Preview enriched VLE interaction events.
student_vle_full.head()

,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type,week_from,week_to,module_presentation_length,relative_time,arab_semester_day
0,AAA,2013J,28400,546652,-10,4,forumng,NaN,NaN,268,-0.037313,-5.597015
1,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,268,-0.037313,-5.597015
2,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,268,-0.037313,-5.597015
3,AAA,2013J,28400,546614,-10,11,homepage,NaN,NaN,268,-0.037313,-5.597015
4,AAA,2013J,28400,546714,-10,1,oucontent,NaN,NaN,268,-0.037313,-5.597015


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The VLE interaction table is now enriched with activity types, course presentation length, normalized course progress, and mapped Arab semester days.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Negative 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">relative_time</code> 
values appear because some VLE interactions happened before the official course start date. These early interactions can still be useful as pre-course engagement signals.
</p>

</div>

<h2 style="color:#EA580C;">📊 Build Aggregated VLE Engagement Features</h2>

This section aggregates VLE interactions into student-level engagement features for each module presentation.

In [5]:
# Aggregate VLE engagement at the student-course-presentation level.
vle_features = (
    student_vle_full.groupby(
        ["id_student", "code_module", "code_presentation"]
    )
    .agg(
        total_clicks=("sum_click", "sum"),
        total_active_days=("date", "nunique"),
        vle_module_presentation_length=("module_presentation_length", "first"),
        unique_sites=("id_site", "nunique"),
        unique_activity_types=("activity_type", "nunique"),
        clicks_first_25=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] <= 0.25].sum()),
        clicks_first_50=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] <= 0.50].sum()),
        clicks_last_25=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] >= 0.75].sum())
    )
    .reset_index()
)

# Calculate the average number of clicks per active day.
vle_features["clicks_per_active_day"] = (
    vle_features["total_clicks"] / vle_features["total_active_days"]
).replace([np.inf, -np.inf], 0).fillna(0)

# Calculate the percentage of course days where the student was active.
vle_features["active_days_ratio"] = (
    vle_features["total_active_days"] / vle_features["vle_module_presentation_length"]
).replace([np.inf, -np.inf], 0).fillna(0)

# Convert active-days ratio to an equivalent 150-day Arab semester scale.
vle_features["arab_active_days_equivalent"] = (
    vle_features["active_days_ratio"] * ARAB_SEMESTER_LENGTH
)

# Preview the generated VLE engagement features.
vle_features.head()

,id_student,code_module,code_presentation,total_clicks,total_active_days,vle_module_presentation_length,unique_sites,unique_activity_types,clicks_first_25,clicks_first_50,clicks_last_25,clicks_per_active_day,active_days_ratio,arab_active_days_equivalent
0,6516,AAA,2014J,2791,159,269,84,7,1118,1603,469,17.553459,0.591078,88.661710
1,8462,DDD,2013J,646,56,261,125,9,527,646,0,11.535714,0.214559,32.183908
2,8462,DDD,2014J,10,1,262,3,3,10,10,0,10.000000,0.003817,0.572519
3,11391,AAA,2013J,934,40,268,55,6,545,710,186,23.350000,0.149254,22.388060
4,23629,BBB,2013B,161,16,240,11,5,119,161,0,10.062500,0.066667,10.000000


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The aggregated VLE features summarize each student's online engagement within a module presentation.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
These features capture total clicks, active days, resource diversity, activity-type diversity, early-course clicks, late-course clicks, and activity intensity.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">active_days_ratio</code> 
and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">arab_active_days_equivalent</code> 
features make engagement easier to compare across courses with different lengths.
</p>

</div>

<h2 style="color:#EA580C;">🧭 Build VLE Activity-Type Features</h2>

This section converts VLE activity types into separate click-based features for each student-course-presentation record.

In [6]:
# Create activity-type features by summing clicks for each VLE activity category.
activity_features = student_vle_full.pivot_table(
    index=["id_student", "code_module", "code_presentation"],
    columns="activity_type",
    values="sum_click",
    aggfunc="sum",
    fill_value=0
).reset_index()

# Remove the pivot column index name for a cleaner table structure.
activity_features.columns.name = None

# Preview generated activity-type click features.
activity_features.head()

,id_student,code_module,code_presentation,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,6516,AAA,2014J,21,0,0,0,451,0,497,...,0,0,0,0,0,0,31,0,143,143
1,8462,DDD,2013J,0,0,12,0,36,0,184,...,0,18,0,0,0,0,70,0,227,23
2,8462,DDD,2014J,0,0,0,0,2,0,7,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,0,0,0,0,193,0,138,...,0,0,0,0,0,0,13,0,32,5
4,23629,BBB,2013B,0,0,0,0,87,0,36,...,0,0,0,0,31,0,2,0,5,0


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The activity-type features show how each student's clicks are distributed across different VLE resources such as forums, homepage, quizzes, resources, subpages, and URLs.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
These features add more detailed engagement signals than total clicks alone, which can help the model distinguish between different learning behaviors.
</p>

</div>

<h2 style="color:#EA580C;">📝 Prepare Assessment Submission Events</h2>

This section enriches student assessment submissions with assessment metadata and course duration, then creates timing and delay features.

In [7]:
# Merge student assessment submissions with assessment metadata.
# This adds assessment type, due date, weight, module, and presentation information.
assessment_full = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

# Add course presentation length to normalize assessment dates.
assessment_full = assessment_full.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

# Convert assessment due date into relative course progress.
assessment_full["assessment_relative_time"] = (
    assessment_full["date"] / assessment_full["module_presentation_length"]
)

# Convert submission date into relative course progress.
assessment_full["submitted_relative_time"] = (
    assessment_full["date_submitted"] / assessment_full["module_presentation_length"]
)

# Map assessment due date to a 150-day Arab semester timeline.
assessment_full["assessment_arab_day"] = (
    assessment_full["assessment_relative_time"] * ARAB_SEMESTER_LENGTH
)

# Map actual submission date to a 150-day Arab semester timeline.
assessment_full["submitted_arab_day"] = (
    assessment_full["submitted_relative_time"] * ARAB_SEMESTER_LENGTH
)

# Identify whether the student submitted after the assessment due date.
assessment_full["is_late"] = (
    assessment_full["date_submitted"] > assessment_full["date"]
)

# Calculate raw submission delay in original OULAD days.
assessment_full["submission_delay"] = (
    assessment_full["date_submitted"] - assessment_full["date"]
)

# Calculate submission delay as a percentage of course duration.
assessment_full["submission_delay_relative"] = (
    assessment_full["submitted_relative_time"] -
    assessment_full["assessment_relative_time"]
)

# Convert relative submission delay into the 150-day Arab semester scale.
assessment_full["submission_delay_arab_days"] = (
    assessment_full["submission_delay_relative"] * ARAB_SEMESTER_LENGTH
)

# Preview enriched assessment submission events.
assessment_full.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight,module_presentation_length,assessment_relative_time,submitted_relative_time,assessment_arab_day,submitted_arab_day,is_late,submission_delay,submission_delay_relative,submission_delay_arab_days
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.067164,10.634328,10.074627,False,-1.0,-0.003731,-0.559701
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.082090,10.634328,12.313433,True,3.0,0.011194,1.679104
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.063433,10.634328,9.514925,False,-2.0,-0.007463,-1.119403
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.097015,10.634328,14.552239,True,7.0,0.026119,3.917910
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.070896,10.634328,10.634328,False,0.0,0.000000,0.000000


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The assessment table now includes timing features that describe when assessments were due, when students submitted them, and whether submissions were late.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
Negative 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">submission_delay</code> 
values mean the student submitted before the due date, while positive values indicate late submission.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The relative and Arab-semester delay features make assessment timing comparable across courses with different lengths.
</p>

</div>

<h2 style="color:#EA580C;">📚 Build Aggregated Assessment Features</h2>

This section aggregates assessment submissions into student-level academic performance and submission-behavior features.

In [8]:
# Aggregate assessment performance and submission behavior
# at the student-course-presentation level.
assessment_features = assessment_full.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    avg_score=("score", "mean"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    submitted_assessments=("id_assessment", "count"),
    late_submissions=("is_late", "sum"),
    avg_submission_delay=("submission_delay", "mean"),
    avg_assessment_relative_time=("assessment_relative_time", "mean"),
    avg_submitted_relative_time=("submitted_relative_time", "mean"),
    avg_assessment_arab_day=("assessment_arab_day", "mean"),
    avg_submitted_arab_day=("submitted_arab_day", "mean"),
    avg_submission_delay_relative=("submission_delay_relative", "mean"),
    avg_submission_delay_arab_days=("submission_delay_arab_days", "mean"),
    assessments_first_50=("assessment_relative_time", lambda x: (x <= 0.50).sum())
).reset_index()

# Preview generated assessment features.
assessment_features.head()

,id_student,code_module,code_presentation,avg_score,min_score,max_score,submitted_assessments,late_submissions,avg_submission_delay,avg_assessment_relative_time,avg_submitted_relative_time,avg_assessment_arab_day,avg_submitted_arab_day,avg_submission_delay_relative,avg_submission_delay_arab_days,assessments_first_50
0,6516,AAA,2014J,61.800000,48.0,77.0,5,0,-2.600000,0.424535,0.414870,63.680297,62.230483,-0.009665,-1.449814,3
1,8462,DDD,2013J,87.666667,83.0,93.0,3,1,-0.333333,0.212005,0.210728,31.800766,31.609195,-0.001277,-0.191571,3
2,8462,DDD,2014J,86.500000,83.0,93.0,4,0,-59.500000,0.223282,-0.003817,33.492366,-0.572519,-0.227099,-34.064885,4
3,11391,AAA,2013J,82.000000,78.0,85.0,5,0,-1.800000,0.426119,0.419403,63.917910,62.910448,-0.006716,-1.007463,3
4,23629,BBB,2013B,82.500000,63.0,100.0,4,3,3.500000,0.217708,0.232292,32.656250,34.843750,0.014583,2.187500,4


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The aggregated assessment features summarize each student's academic performance and submission behavior within a module presentation.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
These features include average score, score range, number of submitted assessments, late submissions, average submission delay, and normalized assessment timing.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">assessments_first_50</code> 
feature captures how many assessments occurred in the first half of the course, which can support early-warning analysis.
</p>

</div>

<h2 style="color:#EA580C;">🧩 Merge All Feature Groups into Final Dataset</h2>

This section combines the base student table with VLE engagement features, VLE activity-type features, and assessment features.

In [9]:
# Merge aggregated VLE engagement features with the base student table.
final_df = base.merge(
    vle_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Add detailed VLE activity-type click features.
final_df = final_df.merge(
    activity_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Add aggregated assessment performance and submission features.
final_df = final_df.merge(
    assessment_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Display the shape of the final merged dataset.
final_df.shape

(32480, 58)

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final merged dataset contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course-presentation records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">58</code> 
columns.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Each row represents one student attempt in a specific module presentation, combining demographic, course, VLE engagement, activity-type, and assessment features.
</p>

</div>

<h2 style="color:#EA580C;">🧼 Handle Missing Values</h2>

This section fills missing numerical feature values with zero and replaces missing deprivation-band values with `Unknown`.

In [10]:
# Select all numerical columns in the final dataset.
numeric_cols = final_df.select_dtypes(include=["number"]).columns

# Fill missing numerical values with zero.
# Missing numerical features usually mean no recorded VLE or assessment activity.
final_df[numeric_cols] = final_df[numeric_cols].fillna(0)

# Fill missing socio-economic deprivation band values with a clear category.
final_df["imd_band"] = final_df["imd_band"].fillna("Unknown")

# Confirm that missing values have been handled.
final_df.isnull().sum().sort_values(ascending=False).head(20)

code_module                       0
code_presentation                 0
id_student                        0
gender                            0
region                            0
highest_education                 0
imd_band                          0
age_band                          0
num_of_prev_attempts              0
studied_credits                   0
disability                        0
final_result                      0
target                            0
module_presentation_length        0
total_clicks                      0
total_active_days                 0
vle_module_presentation_length    0
unique_sites                      0
unique_activity_types             0
clicks_first_25                   0
dtype: int64

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The missing-value check confirms that the main columns now contain no missing values.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
Numerical missing values were filled with 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">0</code> 
because they usually represent missing activity or assessment records. Missing 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">imd_band</code> 
values were kept as an explicit 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">Unknown</code> 
category instead of being removed.
</p>

</div>

<h2 style="color:#EA580C;">⚖️ Create Workload Features</h2>

This section converts `studied_credits` into workload-related features that describe the student's study load intensity.

In [11]:
# Define the standard full-time study load in the OULAD context.
FULL_LOAD_OU = 120

# Calculate the student's workload ratio compared with full-time study load.
final_df["workload_ratio"] = final_df["studied_credits"] / FULL_LOAD_OU

# Limit extreme workload ratio values to avoid very large outlier effects.
final_df["workload_ratio"] = final_df["workload_ratio"].clip(upper=2.0)

# Convert workload ratio into interpretable workload categories.
final_df["workload_level"] = pd.cut(
    final_df["workload_ratio"],
    bins=[0, 0.5, 1.0, 1.5, 2.0],
    labels=["light", "normal", "high", "very_high"],
    include_lowest=True
)

# Preview generated workload features.
final_df[["studied_credits", "workload_ratio", "workload_level"]].head()

,studied_credits,workload_ratio,workload_level
0,240,2.0,very_high
1,60,0.5,light
2,60,0.5,light
3,60,0.5,light
4,60,0.5,light


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The workload features transform raw 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">studied_credits</code> 
into a normalized ratio and an interpretable workload category.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
For example, students with 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">240</code> 
credits are capped at a workload ratio of 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">2.0</code> 
and labeled as 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">very_high</code>, 
while students with 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">60</code> 
credits are labeled as 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">light</code>.
</p>

</div>

<h2 style="color:#EA580C;">🔍 Review Final Dataset Columns</h2>

This section reviews the final column list before saving the processed dataset.

In [12]:
# Display all columns in the final processed dataset.
final_df.columns

Index(['code_module', 'code_presentation', 'id_student', 'gender', 'region',
       'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts',
       'studied_credits', 'disability', 'final_result', 'target',
       'module_presentation_length', 'total_clicks', 'total_active_days',
       'vle_module_presentation_length', 'unique_sites',
       'unique_activity_types', 'clicks_first_25', 'clicks_first_50',
       'clicks_last_25', 'clicks_per_active_day', 'active_days_ratio',
       'arab_active_days_equivalent', 'dataplus', 'dualpane', 'externalquiz',
       'folder', 'forumng', 'glossary', 'homepage', 'htmlactivity',
       'oucollaborate', 'oucontent', 'ouelluminate', 'ouwiki', 'page',
       'questionnaire', 'quiz', 'repeatactivity', 'resource', 'sharedsubpage',
       'subpage', 'url', 'avg_score', 'min_score', 'max_score',
       'submitted_assessments', 'late_submissions', 'avg_submission_delay',
       'avg_assessment_relative_time', 'avg_submitted_relative_time',
   

<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final dataset contains identifiers, demographic fields, target labels, course metadata, VLE engagement features, activity-type click features, assessment performance features, timing features, and workload features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This confirms that the dataset is ready to be saved and used in the next modeling stages.
</p>

</div>

In [13]:
# Quick sanity check for intermediate tables.
print("base:", base.shape)
print("student_vle_full:", student_vle_full.shape)
print("assessment_full:", assessment_full.shape)

base: (32480, 14)
student_vle_full: (10655280, 12)
assessment_full: (173912, 19)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
The intermediate tables were created successfully and have reasonable shapes for the next aggregation and merging steps.
</p>

</div>

<h2 style="color:#EA580C;">💾 Save Final Processed Dataset</h2>

This section saves the final merged and feature-engineered dataset as a processed CSV file for the next modeling notebooks.

In [14]:
from pathlib import Path

# Define the output path for the processed student-course feature dataset.
output_path = Path("data/processed/student_course_features.csv")

# Create the processed data folder if it does not already exist.
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the final machine-learning-ready dataset.
final_df.to_csv(output_path, index=False)

print("Final dataset saved successfully.")
print("Shape:", final_df.shape)

Final dataset saved successfully.
Shape: (32480, 60)


<div style="background:#fff7ed; border:1px solid #fdba74; border-left:5px solid #ea580c; padding:16px 20px; border-radius:12px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<p style="font-size:16px; margin-top:0; color:#c2410c;">
<b>🔎 Observation</b>
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
The final processed dataset was saved successfully as 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">student_course_features.csv</code>.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155;">
It contains 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">32,480</code> 
student-course-presentation records and 
<code style="background:#ffedd5; color:#9a3412; padding:3px 7px; border-radius:6px;">60</code> 
columns after adding workload features.
</p>

<p style="font-size:15.5px; line-height:1.8; color:#334155; margin-bottom:0;">
This file will be used as the main full-course feature dataset for the next early-warning and modeling notebooks.
</p>

</div>

In [16]:
base.to_csv(r"C:\Users\HP\OneDrive\المستندات\GitHub\Student-Leak-Radar\project\archive\data\processed\base_student_table.csv", index=False)
student_vle_full.to_csv(r"C:\Users\HP\OneDrive\المستندات\GitHub\Student-Leak-Radar\project\archive\data\processed\student_vle_full.csv", index=False)
assessment_full.to_csv(r"C:\Users\HP\OneDrive\المستندات\GitHub\Student-Leak-Radar\project\archive\data\processed\assessment_full.csv", index=False)

<div style="background:#f8fafc; border:1px solid #dbeafe; border-left:7px solid #2563eb; padding:26px 30px; border-radius:16px; font-family:Segoe UI, Arial, sans-serif; color:#1e293b;">

<h2 style="margin-top:0; color:#1d4ed8; font-size:30px;">
✅ Notebook Summary
</h2>

<p style="font-size:17px; line-height:1.8; color:#334155;">
This notebook successfully created the full-course processed dataset 
<code style="background:#e0f2fe; color:#075985; padding:3px 7px; border-radius:6px;">student_course_features.csv</code>.
</p>

<hr style="border:0; border-top:1px solid #bfdbfe; margin:24px 0;">

<h2 style="color:#0f172a; font-size:26px;">
🧩 Workflow Completed
</h2>

<div style="background:#ffffff; border:1px solid #bfdbfe; border-radius:14px; padding:18px 22px; margin:14px 0 22px 0; box-shadow:0 2px 8px rgba(37,99,235,0.08);">

<ul style="font-size:16px; line-height:2; margin:0;">
  <li>Loading the raw OULAD tables</li>
  <li>Preparing the base student table</li>
  <li>Enriching VLE interaction events</li>
  <li>Building aggregated VLE engagement features</li>
  <li>Creating VLE activity-type features</li>
  <li>Preparing assessment submission events</li>
  <li>Building aggregated assessment features</li>
  <li>Merging all feature groups into one flat dataset</li>
  <li>Handling missing values</li>
  <li>Creating workload features</li>
  <li>Reviewing final columns</li>
  <li>Saving the processed dataset</li>
</ul>

</div>

<p style="font-size:17px; line-height:1.8; color:#334155;">
The output dataset contains <b>one row per student-course-presentation attempt</b> and will be used as the main input for the next early-warning and modeling notebooks.
</p>

</div>